# Archival SBC notebook

This notebook preserves the supplied original diagnostic, which sampled labeled A values separately from the prediction pool. The final manuscript uses labeled examples drawn without replacement from the observed pool. Run `python code/ppi/sbc.py` from the repository root (or `make sbc`) for the corrected design and final reference diagnostics. See `REPRODUCIBILITY.md`.


# Simulation-based calibration for conjugate CRE, revised notebook

This notebook keeps the original SBC idea but makes it more notebook-friendly and reproducible:

- uses `np.random.Generator` throughout
- saves rank histograms and a CSV summary
- can run both Uniform and Jeffreys priors
- treats \(g = \theta_A\theta_{H|1} + (1-\theta_A)\theta_{H|0}\) as a derived quantity

The SBC is synthetic. The truth \(g\) here is generated from the prior, not from the ADNI cohort.

In [ ]:
from pathlib import Path
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


In [ ]:
%pip install -q numpy pandas scipy matplotlib tqdm

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm
from scipy.stats import chisquare, kstest

In [ ]:
# ============================================================
# 0) Settings
# ============================================================

OUT_DIR = REPO_ROOT / "results/sbc"
FIG_DIR = OUT_DIR / "figures"

OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# M = number of simulated datasets
# S = posterior draws per dataset
M = 500
S = 1000

# Synthetic sample sizes for SBC.
# These do not need to match ADNI exactly, but NA=2116 reflects the revised full cohort size.
NA = 2116
NH = 100

BINS = 20
SEED = 2025

PRIORS_TO_RUN = ["jeffreys", "uniform"]

print("OUT_DIR:", OUT_DIR)
print("M:", M, "S:", S, "NA:", NA, "NH:", NH)

In [ ]:
# ============================================================
# 1) SBC utilities
# ============================================================

def prior_params(prior: str):
    prior = prior.lower()
    if prior == "jeffreys":
        return {
            "aA": 0.5, "bA": 0.5,
            "a1": 0.5, "b1": 0.5,
            "a0": 0.5, "b0": 0.5,
        }
    if prior == "uniform":
        return {
            "aA": 1.0, "bA": 1.0,
            "a1": 1.0, "b1": 1.0,
            "a0": 1.0, "b0": 1.0,
        }
    raise ValueError(f"Unknown prior: {prior}")


def draw_beta_post(alpha: float, beta: float, size: int, rng: np.random.Generator) -> np.ndarray:
    return rng.beta(alpha, beta, size=size)


def rank_stat(draws: np.ndarray, truth: float) -> int:
    return int(np.sum(draws < truth))


def run_sbc_cre(
    M: int,
    S: int,
    NA: int,
    NH: int,
    prior: str,
    seed: int,
):
    rng = np.random.default_rng(seed)
    pp = prior_params(prior)

    ranks_thetaA = np.empty(M, dtype=int)
    ranks_thetaH1 = np.empty(M, dtype=int)
    ranks_thetaH0 = np.empty(M, dtype=int)
    ranks_g = np.empty(M, dtype=int)

    truth_rows = []

    for m in tqdm(range(M), desc=f"SBC {prior}"):
        thetaA_star = rng.beta(pp["aA"], pp["bA"])
        thetaH1_star = rng.beta(pp["a1"], pp["b1"])
        thetaH0_star = rng.beta(pp["a0"], pp["b0"])

        g_star = thetaA_star * thetaH1_star + (1.0 - thetaA_star) * thetaH0_star

        # Full autorater pool
        A_unlab = rng.binomial(1, thetaA_star, size=NA)

        # Labeled subset
        labeled_idx = rng.choice(NA, size=NH, replace=False)
        A_lab = A_unlab[labeled_idx]
        H_lab = np.where(
            A_lab == 1,
            rng.binomial(1, thetaH1_star, size=NH),
            rng.binomial(1, thetaH0_star, size=NH),
        )

        nA = int(A_unlab.sum())

        n11 = int(np.sum((A_lab == 1) & (H_lab == 1)))
        n10 = int(np.sum((A_lab == 1) & (H_lab == 0)))
        n01 = int(np.sum((A_lab == 0) & (H_lab == 1)))
        n00 = int(np.sum((A_lab == 0) & (H_lab == 0)))

        alphaA = pp["aA"] + nA
        betaA = pp["bA"] + (NA - nA)

        alpha1 = pp["a1"] + n11
        beta1 = pp["b1"] + n10

        alpha0 = pp["a0"] + n01
        beta0 = pp["b0"] + n00

        thetaA_draws = draw_beta_post(alphaA, betaA, S, rng)
        thetaH1_draws = draw_beta_post(alpha1, beta1, S, rng)
        thetaH0_draws = draw_beta_post(alpha0, beta0, S, rng)

        g_draws = thetaA_draws * thetaH1_draws + (1.0 - thetaA_draws) * thetaH0_draws

        ranks_thetaA[m] = rank_stat(thetaA_draws, thetaA_star)
        ranks_thetaH1[m] = rank_stat(thetaH1_draws, thetaH1_star)
        ranks_thetaH0[m] = rank_stat(thetaH0_draws, thetaH0_star)
        ranks_g[m] = rank_stat(g_draws, g_star)

        truth_rows.append({
            "rep": m,
            "thetaA_star": thetaA_star,
            "thetaH1_star": thetaH1_star,
            "thetaH0_star": thetaH0_star,
            "g_star": g_star,
            "nA": nA,
            "n11": n11,
            "n10": n10,
            "n01": n01,
            "n00": n00,
        })

    ranks = {
        "thetaA": ranks_thetaA,
        "thetaH1": ranks_thetaH1,
        "thetaH0": ranks_thetaH0,
        "g": ranks_g,
    }

    truth_df = pd.DataFrame(truth_rows)
    return ranks, truth_df

In [ ]:
# ============================================================
# 2) Plotting and test utilities
# ============================================================

def rank_to_u(ranks: np.ndarray, S: int) -> np.ndarray:
    return (ranks + 0.5) / (S + 1.0)


def summarize_rank_uniformity(ranks: np.ndarray, S: int, B: int):
    u = rank_to_u(ranks, S)
    bins = np.linspace(0.0, 1.0, B + 1)
    counts, _ = np.histogram(u, bins=bins)
    expected = np.full(B, len(u) / B)

    chi2_stat, chi2_p = chisquare(counts, expected)
    ks_stat, ks_p = kstest(u, "uniform")

    return {
        "n_ranks": len(ranks),
        "n_bins": B,
        "expected_per_bin": len(ranks) / B,
        "chi2_stat": float(chi2_stat),
        "chi2_p": float(chi2_p),
        "ks_stat": float(ks_stat),
        "ks_p": float(ks_p),
        "min_count": int(counts.min()),
        "max_count": int(counts.max()),
    }


def plot_rank_histogram(ranks: np.ndarray, S: int, title: str, out_path: Path, B: int):
    u = rank_to_u(ranks, S)
    bins = np.linspace(0.0, 1.0, B + 1)
    counts, _ = np.histogram(u, bins=bins)
    expected = len(u) / B

    stats = summarize_rank_uniformity(ranks, S, B)

    plt.figure(figsize=(5.5, 3.6))
    xs = 0.5 * (bins[1:] + bins[:-1])
    width = 1.0 / B

    plt.bar(xs, counts, width=width * 0.9, align="center")
    plt.hlines(expected, 0, 1, linestyles="dashed", linewidth=1.2)

    plt.xlim(0, 1)
    plt.xlabel("Normalized rank")
    plt.ylabel("Count per bin")
    plt.title(title)

    txt = f"chi2 p={stats['chi2_p']:.2f}; KS p={stats['ks_p']:.2f}"
    plt.text(0.99, 0.97, txt, ha="right", va="top", transform=plt.gca().transAxes, fontsize=9)

    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()

    return stats

In [ ]:
# ============================================================
# 3) Run SBC for selected priors
# ============================================================

all_summary_rows = []

for prior in PRIORS_TO_RUN:
    prior_out = OUT_DIR / prior
    prior_fig = FIG_DIR / prior
    prior_out.mkdir(parents=True, exist_ok=True)
    prior_fig.mkdir(parents=True, exist_ok=True)

    ranks, truth_df = run_sbc_cre(
        M=M,
        S=S,
        NA=NA,
        NH=NH,
        prior=prior,
        seed=SEED + (0 if prior == "jeffreys" else 100000),
    )

    truth_csv = prior_out / f"sbc_truth_counts_{prior}.csv"
    truth_df.to_csv(truth_csv, index=False)
    print("Saved:", truth_csv)

    for param_name, param_ranks in ranks.items():
        fig_path = prior_fig / f"sbc_{param_name}_{prior}_b{BINS}.png"
        stats = plot_rank_histogram(
            ranks=param_ranks,
            S=S,
            title=f"SBC rank histogram: {param_name} ({prior})",
            out_path=fig_path,
            B=BINS,
        )

        stats.update({
            "prior": prior,
            "parameter": param_name,
            "M": M,
            "S": S,
            "NA": NA,
            "NH": NH,
            "figure": str(fig_path),
        })

        all_summary_rows.append(stats)

summary = pd.DataFrame(all_summary_rows)
summary_csv = OUT_DIR / "sbc_uniformity_summary.csv"
summary.to_csv(summary_csv, index=False)

print("Saved:", summary_csv)
display(summary.round(4))

In [ ]:
# ============================================================
# 4) Compact paper-friendly table
# ============================================================

paper_sbc = summary[
    [
        "prior", "parameter", "chi2_p", "ks_p",
        "min_count", "max_count", "expected_per_bin",
        "M", "S", "NA", "NH",
    ]
].copy()

paper_csv = OUT_DIR / "sbc_paper_summary.csv"
paper_sbc.to_csv(paper_csv, index=False)

display(paper_sbc.round(3))
print("Saved:", paper_csv)